In [0]:
import json
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import DataFrame, SparkSession, Row
from typing import List

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


class TransformationConfigError(Exception):
    """Raised after all transformations are attempted, if one or more failed."""
    pass


def load_transformation_config(spark: SparkSession, job_id: int) -> List[Row]:
    """
    Load transformation rules for a given job, ordered by Transformation_id.

    Args:
        spark: Active SparkSession used to run the query.
        job_id: The job these transformation rules apply to
            (matches the Job_Id column in TransformationConfig).

    Returns:
        List of Row objects, each exposing Transformation_id, Job_Id,
        Transformation_Name, Transformation_Arguments (JSON string),
        and Column_Name, ordered by Transformation_id.
    """
    return spark.sql(f"""
        SELECT * FROM ct_oil_gas.sc_metadata.TransformationConfig
        WHERE Job_Id = {job_id}
        ORDER BY Transformation_id
    """).collect()


class Transformation:
    """
    Applies configured transformations to a DataFrame.

    Behavior is driven by a transformation config (rows from
    TransformationConfig) rather than hardcoded column logic, so new
    columns, casts, or cleanup rules can be added without code changes.
    Failures during apply_config are logged and counted rather than
    raised immediately, so one bad rule doesn't stop the rest of the
    transformations from running.
    """

    def __init__(self, df: DataFrame):
        """
        Args:
            df: The DataFrame to transform. Mutated in place across
                method calls via self.df.
        """
        self.df = df

    def drop_duplicates(self) -> DataFrame:
        """Drop fully duplicate rows across all columns."""
        self.df = self.df.dropDuplicates()
        return self.df

    def drop_nulls(self) -> DataFrame:
        """Drop rows containing any null values."""
        self.df = self.df.dropna()
        return self.df

    def date_cast(self,col_name)-> DataFrame:
        """
        Cast the Timestamp to onlt date format
        """
        self.df= self.df.withColumn(col_name, F.to_date(F.col(col_name)))

        return self.df

    def deduplicate_by_key(self, key_columns: List[str], order_by_column: str) -> DataFrame:
        """
        Keep only the first occurrence per key_columns, ordered by order_by_column ascending.

        Args:
            key_columns: Column(s) to partition duplicates by (e.g. ["transaction_id"]).
            order_by_column: Column to order within each partition to decide
                which row is "first" (e.g. ingestion_timestamp).

        Returns:
            The deduplicated DataFrame.
        """
        w = Window.partitionBy(*key_columns).orderBy(order_by_column)
        self.df = (
            self.df.withColumn("row_num", F.row_number().over(w))
            .filter(F.col("row_num") == 1)
            .drop("row_num")
        )
        return self.df


    def apply_cast(self, column_name: str, target_type: str) -> DataFrame:
        """Cast a single column to the given Spark SQL type string."""
        self.df = self.df.withColumn(column_name, F.col(column_name).cast(target_type))
        return self.df

    def apply_trim(self, column_name: str) -> DataFrame:
        """Trim whitespace on a single string column."""
        self.df = self.df.withColumn(column_name, F.trim(F.col(column_name)))
        return self.df

    def apply_config(self, config_rows: List[Row]) -> DataFrame:
        """
        Apply a sequence of configured transformations to self.df.


        """
        error_count = 0
        errors = []

        for row in config_rows:
            ttype = row["Transformation_Name"]
            col_name = row["Column_Name"]

            try:
                args = json.loads(row["Transformation_Arguments"] or "{}")

                if ttype == "CAST":
                    self.apply_cast(col_name, args["type"])
                elif ttype == "TRIM":
                    self.apply_trim(col_name)
                elif ttype == "DEDUP_KEY":
                    key_columns = [c.strip() for c in col_name.split(",")]
                    self.deduplicate_by_key(key_columns, args["order_by"])
                elif ttype == "DROP_NULLS":
                    self.drop_nulls()
                elif ttype == "DROP_DUPLICATES":
                    self.drop_duplicates()
                elif ttype =="date_cast":
                    self.date_cast(col_name)
                else:
                    raise ValueError(f"Unknown Transformation_Name: {ttype}")

            except Exception as e:
                error_count += 1
                error_msg = (
                    f"Failed transformation [name={ttype}, column={col_name}]: {e}"
                )
                errors.append(error_msg)
                logger.error(error_msg)

        if error_count > 0:
            summary = f"{error_count} transformation(s) failed:\n" + "\n".join(errors)
            raise TransformationConfigError(summary)

        return self.df

In [0]:
# catalog_name = "ct_oil_gas"
# bronze_schema = "sc_bronze"
# table_name = "oil_gas_transactions"

# config_rows = load_transformation_config(spark,1)

# df = spark.table(f"{catalog_name}.{bronze_schema}.oil_gas_transactions")

# transformer = Transformation(df)
# transformed_df = transformer.apply_config(config_rows)